In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
def log_pipeline_error(step_name, error):
    spark.createDataFrame(
        [(notebook_name, step_name, str(error), datetime.now())],
        ["notebook", "step", "error_message", "error_time"]
    ).write.mode("append").saveAsTable("dev_catalog.default.pipeline_errors")

In [0]:
category = {
    "TA" :  "Class A Trunk Road",
    "TM" : "Class A Trunk Motorway",
    "PA" : "Class A Principal Road",
    "PM" : "Class A Principal Motorway",
    "M"  : "Class B Road"
}
notebook_name = "04_silver_roads_batch"
 


In [0]:
try:
    df_roads = spark.table("dev_catalog.bronze.raw_roads")
except Exception as e:
    log_pipeline_error("reading bronze raw roads file", e)
    raise

In [0]:
try:
    df_roads = (
        df_roads
        .withColumn("count_point_id", col("count_point_id").try_cast(IntegerType()))
        .withColumn("year", col("year").try_cast(IntegerType()))
        .withColumn("road_category" , when(~col("road_category").isin(list(category.keys())), lit(None)).otherwise(col("road_category")))
        .withColumn("link_length_km", col("link_length_km").try_cast(DoubleType()))
        .withColumn("link_length_miles", col("link_length_miles").try_cast(DoubleType()))
    )
except Exception as e:
    log_pipeline_error("transforming roads silver file phase1",e)
    raise

In [0]:

try:
    map_expression = create_map([lit(x) for pair in category.items() for x in pair])

    df_final = (
        df_roads
        .dropDuplicates(["count_point_id"])
        .dropna(how="all")
        .withColumn("source" , ifnull(col("source"),lit("A")))
        .withColumn("road_category_name", map_expression[col('road_category')])
        .withColumn("expected_road_type" , when(col("road_category_name").contains("Class A") , lit("Major"))
                                            .when (col("road_category_name").contains("Class B") , lit("Minor"))
                                            .otherwise(lit("Unknown")))
        .withColumn("road_type_consistent", when(col("road_type") == col("expected_road_type"), lit(True))
                                            .otherwise(lit(False)))
        .withColumn("transformed_Time", current_timestamp())
        .withColumn("valid_from", current_timestamp())
        )
except Exception as e:
    log_pipeline_error("transforming roads silver file phase2",e)
    raise


In [0]:
df_final.createOrReplaceTempView("final_silver")

SCD2

In [0]:
%sql
MERGE INTO dev_catalog.silver.roads AS target
USING final_silver AS src
ON target.count_point_id = src.count_point_id and target.source = src.source
WHEN MATCHED THEN UPDATE SET target.valid_to = current_timestamp()

In [0]:
%sql
MERGE INTO dev_catalog.silver.roads AS target
USING final_silver AS src
ON target.count_point_id = src.count_point_id and target.source = src.source
WHEN NOT MATCHED THEN INSERT (count_point_id, road_type_consistent, year, region_id, region_name, region_ons_code, local_authority_id, local_authority_name, local_authority_code, road_name, road_category, road_type, start_junction_road_name, end_junction_road_name, easting, northing, latitude, longitude, link_length_km, link_length_miles, source, valid_from, transformed_Time)
VALUES (src.count_point_id, src.road_type_consistent, src.year, src.region_id, src.region_name, src.region_ons_code, src.local_authority_id, src.local_authority_name, src.local_authority_code, src.road_name, src.road_category, src.road_type, src.start_junction_road_name, src.end_junction_road_name, src.easting, src.northing, src.latitude, src.longitude, src.link_length_km, src.link_length_miles, src.source, current_timestamp(), current_timestamp())